# 09 — RelMS: All 5 Image Types + Clinical (complete cases only)

Combines all 5 image-type PCA blocks with clinical variables in **one** feature matrix
and computes a single RelMS distance matrix with **6 explicit blocks**:

| Block | Columns | p |
|-------|---------|---|
| Imaging block 1 | 110 PCA — coronal | 110 |
| Imaging block 2 | 110 PCA — sagittal GFC | 110 |
| Imaging block 3 | 110 PCA — transverse GFC | 110 |
| Imaging block 4 | 110 PCA — masked transverse | 110 |
| Imaging block 5 | 110 PCA — native sagittal | 110 |
| Clinical quant block 6 | Age, MMSE, eTIV, nWBV, ASF | 5 |
| Binary (`p2=1`) | M/F encoded as 1/0 | 1 |
| Multiclass (`p3=2`) | Educ, SES | 2 |

**`p1 = [110, 110, 110, 110, 110, 5]`** — each block is normalised by its own
geometric variability before combining, so no single block dominates.

**Missing values are dropped — no imputation.** Only patients with complete
records for Educ, SES and MMSE are kept.

## Install and imports

In [6]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-400:] if r.stderr else "(no stderr)")

run_pip("robust-mixed-dist==0.1.26")
for pkg in ["kmedoids", "openpyxl"]:
    run_pip(pkg)
print("Done.")

import importlib.metadata
print(f"robust-mixed-dist version: {importlib.metadata.version('robust-mixed-dist')}")

  OK    robust-mixed-dist==0.1.26
  OK    kmedoids
  OK    openpyxl
Done.
robust-mixed-dist version: 0.1.26


In [7]:
import os
import numpy as np
import pandas as pd

import kmedoids
# function lives in robust_mixed_dist.mixed — not at the top-level package
from robust_mixed_dist.mixed import related_metric_scaling_dist_matrix_faster
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 1 — Load and filter clinical data

Drop any patient with a missing value in **Educ, SES or MMSE**.  
No imputation. CDR is kept only for post-hoc evaluation.

In [8]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

# Normalise patient IDs: strip _MR1/_MR2 suffix
df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)

# Deduplicate: if a patient has two rows keep the one with a CDR value
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)

# Columns used as features (CDR excluded)
DROP_IF_MISSING = ["Educ", "SES", "MMSE"]
FEATURE_COLS    = ["Age", "M/F", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"]
KEEP            = ["patient_id"] + FEATURE_COLS + ["CDR"]
df_clin         = df_clin[KEEP].copy()

n_before = len(df_clin)
df_clin  = df_clin.dropna(subset=DROP_IF_MISSING).reset_index(drop=True)
n_after  = len(df_clin)

print(f"Patients before dropping : {n_before}")
print(f"Patients after  dropping : {n_after}  (dropped {n_before - n_after})")
print(f"Patients with CDR        : {df_clin['CDR'].notnull().sum()} / {n_after}")

complete_ids = set(df_clin["patient_id"])

Patients before dropping : 416
Patients after  dropping : 216  (dropped 200)
Patients with CDR        : 216 / 216


## Step 2 — Load and filter all 5 PCA files

For each file: drop `patient_id`, keep only the patients identified in Step 1,
and align rows to the same order as `df_clin`.

In [9]:
PCA_FILES = [
    ("cor",        "pca_cor.csv"),
    ("gfc_sag",    "pca_gfc_sag.csv"),
    ("gfc_tra",    "pca_gfc_tra.csv"),
    ("masked_tra", "pca_masked_tra.csv"),
    ("sbj_sag",    "pca_sbj_sag.csv"),
]

pca_blocks = {}   # image_type → np.ndarray (n_complete, 110)

for itype, fname in PCA_FILES:
    df = pd.read_csv(os.path.join(NB_DIR, fname))
    # Align to df_clin row order via a left-join on patient_id
    df_aligned = df_clin[["patient_id"]].merge(df, on="patient_id", how="left")
    pca_cols   = [c for c in df_aligned.columns if c != "patient_id"]
    pca_blocks[itype] = df_aligned[pca_cols].to_numpy(dtype=np.float64)
    print(f"  {itype:<14} block shape: {pca_blocks[itype].shape}  "
          f"NaN in block: {np.isnan(pca_blocks[itype]).sum()}")

print(f"\nAll 5 blocks filtered to {n_after} complete patients.")

  cor            block shape: (216, 110)  NaN in block: 0
  gfc_sag        block shape: (216, 110)  NaN in block: 0
  gfc_tra        block shape: (216, 110)  NaN in block: 0
  masked_tra     block shape: (216, 110)  NaN in block: 0
  sbj_sag        block shape: (216, 110)  NaN in block: 0

All 5 blocks filtered to 216 complete patients.


## Step 3 — Build the combined feature matrix X

Column order (required by RelMS):
```
[cor ×110] [gfc_sag ×110] [gfc_tra ×110] [masked_tra ×110] [sbj_sag ×110]
[Age MMSE eTIV nWBV ASF ×5]  [MF ×1]  [Educ SES ×2]
```
Expected shape: `(n_patients, 558)`.

In [10]:
# Encode clinical columns
df_c = df_clin.copy()
df_c["MF"]   = (df_c["M/F"] == "M").astype(int)
df_c["Educ"] = df_c["Educ"].astype(int)
df_c["SES"]  = df_c["SES"].astype(int)

QUANT_CLIN = ["Age", "MMSE", "eTIV", "nWBV", "ASF"]

X = np.concatenate([
    pca_blocks["cor"],                                         # 110
    pca_blocks["gfc_sag"],                                     # 110
    pca_blocks["gfc_tra"],                                     # 110
    pca_blocks["masked_tra"],                                  # 110
    pca_blocks["sbj_sag"],                                     # 110
    df_c[QUANT_CLIN].to_numpy(dtype=np.float64),               #   5
    df_c[["MF"]].to_numpy(dtype=np.float64),                   #   1
    df_c[["Educ", "SES"]].to_numpy(dtype=np.float64),          #   2
], axis=1)

patient_ids = df_c["patient_id"].values

print(f"X shape           : {X.shape}  — expected ({n_after}, 558)")
print(f"NaN values in X   : {np.isnan(X).sum()}")
assert X.shape[1] == 558, f"Column count mismatch: got {X.shape[1]}, expected 558"

X shape           : (216, 558)  — expected (216, 558)
NaN values in X   : 0


## Step 4 — Compute combined distance matrix

`p1=[110, 110, 110, 110, 110, 5]` tells RelMS to treat each image type and the
clinical quantitative variables as **separate blocks**, normalising each one
independently before combining.

Note: the function is passed as a DataFrame to work around a bug in 0.1.26
where numpy array inputs leave `X_arr` unassigned.

In [11]:
print("Computing RelMS distance matrix ...")

D = related_metric_scaling_dist_matrix_faster(
    X=pd.DataFrame(X),           # DataFrame required — avoids numpy bug in 0.1.26
    p1=[110, 110, 110, 110, 110, 5],
    p2=1,
    p3=2,
    d1="euclidean",
    d2="sokal",
    d3="hamming",
    weights=None,
)

print(f"D shape           : {D.shape}")
print(f"Value range       : [{D.min():.4f}, {D.max():.4f}]")
print(f"NaN values in D   : {np.isnan(D).sum()}  — {'OK, no NaN' if np.isnan(D).sum()==0 else 'WARNING: NaN present'}")
print(f"Is symmetric      : {np.allclose(D, D.T)}")

Computing RelMS distance matrix ...
D shape           : (216, 216)
Value range       : [0.0000, 8.7989]
NaN values in D   : 0  — OK, no NaN
Is symmetric      : True


C:\Users\aregk\AppData\Local\Programs\Python\Python312\Lib\site-packages\robust_mixed_dist\mixed.py:1002: UserWarning: Gram matrix for block 8 is not PSD (min eig=-1.47e+01). Transformation applied.
  warnings.warn(


## Step 5 — Cluster using kmedoids.fasterpam

D is already computed, so `kmedoids.fasterpam` is called directly
(not via `SampleDistClustering`, which does not accept precomputed matrices).

In [12]:
cluster_labels = {}

for k in [3, 4]:
    result = kmedoids.fasterpam(D, medoids=k, random_state=42)
    cluster_labels[k] = np.array(result.labels, dtype=int)
    sizes = {c: int((cluster_labels[k] == c).sum()) for c in sorted(set(cluster_labels[k]))}
    print(f"k={k}  loss={result.loss:.4f}  sizes={sizes}")

k=3  loss=1458.9306  sizes={np.int64(0): 80, np.int64(1): 66, np.int64(2): 70}
k=4  loss=1445.9995  sizes={np.int64(0): 74, np.int64(1): 42, np.int64(2): 68, np.int64(3): 32}


## Step 6 — Evaluate results

Since only complete-case patients were kept, every patient in X has a CDR value.
CDR is multiplied by 2 to convert to integers for ARI and NMI.

In [13]:
cdr_values = df_c["CDR"].values
mask_cdr   = pd.Series(cdr_values).notnull().values
cdr_int    = (cdr_values[mask_cdr] * 2).astype(int)

print(f"Total patients    : {len(patient_ids)}")
print(f"Patients with CDR : {mask_cdr.sum()} — {'all patients have CDR' if mask_cdr.all() else f'{(~mask_cdr).sum()} patients are missing CDR'}")
print(f"CDR categories    : {sorted(set(cdr_int))}  (CDR × 2)")

all_results = []

for k in [3, 4]:
    labels = cluster_labels[k]

    # ── CDR crosstab ──────────────────────────────────────────────────────────
    df_ev  = pd.DataFrame({"cluster": labels, "CDR": cdr_values})
    df_ev  = df_ev[df_ev["CDR"].notnull()].copy()
    df_ev["CDR"] = df_ev["CDR"].astype(str)

    ct = pd.crosstab(df_ev["cluster"], df_ev["CDR"],
                     margins=True, margins_name="Total")
    pct = pd.crosstab(df_ev["cluster"], df_ev["CDR"],
                      normalize="index").mul(100).round(1)

    print(f"\n{'─'*55}")
    print(f"  k={k}")
    print(f"{'─'*55}")
    print("Count:")
    print(ct.to_string())
    print("\nRow % (CDR proportion within each cluster):")
    print(pct.to_string())

    # ── Spread ───────────────────────────────────────────────────────────────
    pct_tab = pd.crosstab(df_ev["cluster"], df_ev["CDR"], normalize="index").mul(100)
    pct_h   = pct_tab["0.0"] if "0.0" in pct_tab.columns \
              else pd.Series(0.0, index=pct_tab.index)
    spread  = float(pct_h.max() - pct_h.min())

    # ── Silhouette ────────────────────────────────────────────────────────────
    sil = silhouette_score(D, labels, metric="precomputed")

    # ── ARI / NMI ─────────────────────────────────────────────────────────────
    ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
    nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

    print(f"\n  Spread  : {spread:.2f}%")
    print(f"  Sil     : {sil:.4f}")
    print(f"  ARI     : {ari:.4f}")
    print(f"  NMI     : {nmi:.4f}")

    all_results.append({
        "method":             "relms_p1=[110,110,110,110,110,5]",
        "k":                  k,
        "n_patients":         len(patient_ids),
        "silhouette":         round(sil, 4),
        "ari":                round(ari, 4),
        "nmi":                round(nmi, 4),
        "spread_pct_healthy": round(spread, 2),
    })

Total patients    : 216
Patients with CDR : 216 — all patients have CDR
CDR categories    : [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]  (CDR × 2)

───────────────────────────────────────────────────────
  k=3
───────────────────────────────────────────────────────
Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         49   23    8    0     80
1         35   23    7    1     66
2         49   11    9    1     70
Total    133   57   24    2    216

Row % (CDR proportion within each cluster):
CDR       0.0   0.5   1.0  2.0
cluster                       
0        61.3  28.7  10.0  0.0
1        53.0  34.8  10.6  1.5
2        70.0  15.7  12.9  1.4

  Spread  : 16.97%
  Sil     : 0.0159
  ARI     : 0.0092
  NMI     : 0.0206

───────────────────────────────────────────────────────
  k=4
───────────────────────────────────────────────────────
Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         46   20    8    0     74


## Step 7 — Save results

In [14]:
df_results = pd.DataFrame(all_results)

print("RelMS  p1=[110,110,110,110,110,5]  —  all 5 image types + clinical")
print("=" * 65)
print(df_results.to_string(index=False))

out_path = os.path.join(NB_PADDED, "results_combined_clinical_image_new.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

RelMS  p1=[110,110,110,110,110,5]  —  all 5 image types + clinical
                          method  k  n_patients  silhouette    ari    nmi  spread_pct_healthy
relms_p1=[110,110,110,110,110,5]  3         216      0.0159 0.0092 0.0206               16.97
relms_p1=[110,110,110,110,110,5]  4         216      0.0112 0.0202 0.0268               20.59

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_combined_clinical_image_new.csv
